## Imports 

In [ ]:
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

TARGET = 'addicted_label'
ID_COL = 'id'
RANDOM_STATE = 42

## Prparing categorical features

In [12]:
categorical_features = ["gender", "stress_level", "academic_work_impact"]

for col in categorical_features:
    train[col] = train[col].fillna("Missing").astype(str)
    test[col] = test[col].fillna("Missing").astype(str)

    # Fit encoding on combined train+test categories so codes match
    combined = pd.concat([train[col], test[col]], axis=0)
    categories = combined.astype("category").cat.categories

    train[col] = pd.Categorical(train[col], categories=categories).codes
    test[col] = pd.Categorical(test[col], categories=categories).codes


print(train[categorical_features].dtypes)
print(test[categorical_features].dtypes)

gender                  int8
stress_level            int8
academic_work_impact    int8
dtype: object
gender                  int8
stress_level            int8
academic_work_impact    int8
dtype: object


In [3]:
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## Train LightGBM baseline

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010502 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1957
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.951602	valid_0's binary_logloss: 0.25626
[200]	valid_0's auc: 0.958767	valid_0's binary_logloss: 0.235964
[300]	valid_0's auc: 0.960465	valid_0's binary_logloss: 0.230726
[400]	valid_0's auc: 0.961125	valid_0's binary_logloss: 0.228639
[500]	valid_0's auc: 0.96145	valid_0's binary_logloss: 0.227575
[600]	valid_0's auc: 0.961724	valid_0's binary_logloss: 0

,num_leaves,64
,learning_rate,0.05
,n_estimators,1000
,random_state,42
,n_jobs,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


## Validation AUC check

In [6]:
val_preds = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_preds)
print(f"Validation ROC AUC: {val_auc:.5f}")

Validation ROC AUC: 0.96236


## Feature importance

In [7]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

notifications_per_day      9256
app_opens_per_day          8552
daily_screen_time_hours    7584
weekend_screen_time        7195
social_media_hours         6705
work_study_hours           6448
gaming_hours               6093
sleep_hours                5269
age                        2901
stress_level                724
gender                      699
academic_work_impact        440
dtype: int32


## Retrain on full training data

In [8]:
final_model = lgb.LGBMClassifier(
    n_estimators=model.best_iteration_,
    learning_rate=0.05,
    num_leaves=64,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
final_model.fit(X, y)

[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010183 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1955
[LightGBM] [Info] Number of data points in the train set: 691369, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892590
[LightGBM] [Info] Start training from score 0.892590


,num_leaves,64
,learning_rate,0.05
,n_estimators,982
,random_state,42
,n_jobs,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


## Predict on test data

In [10]:
test_preds = final_model.predict_proba(X_test)[:, 1]

## Submission

In [11]:
submission = sample_submission.copy()
submission[TARGET] = test_preds
submission.to_csv("submission.csv", index=False)
submission.head()

,id,addicted_label
0,691369,0.999052
1,691370,0.966976
2,691371,0.941132
3,691372,0.984558
4,691373,0.997511
